
# Supplementary figures - per-model layer detail

Fig. 1, 2, 4 and 5 of the main text summarise each model by its **best** layer. These
supplementary figures show the full per-layer picture behind that summary:

* one bar chart per model x metric (ARI / P@5 / silhouette across all layers);
* one bar chart per model x unseen virus (Precision@5 across all layers).

The same two-stage layout is used as in `main_figures.ipynb`: Stage 1 writes the numbers
to two committed CSVs under `results/figure_data/supplementary/`; Stage 2 renders one
PNG+PDF per model into `results/figures/supplementary/`.

In [1]:
# Make figure_io importable whether Jupyter is launched from the repo root or
# from the visualization/ folder, then apply the shared plotting style.
import sys, pathlib
_here = pathlib.Path.cwd()
if not (_here / "figure_io.py").exists() and (_here / "visualization" / "figure_io.py").exists():
    _here = _here / "visualization"
sys.path.insert(0, str(_here))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import figure_io as fio
from figure_io import MODELS, DISPLAY, MODEL_COLOR, save_table, load_table, save_figure

fio.set_style()
print("repository root :", fio.ROOT)
print("reading metrics :", fio.DATA_METRICS.relative_to(fio.ROOT).as_posix())
print("writing results :", fio.RESULTS.relative_to(fio.ROOT).as_posix())

repository root : C:\Users\aleks\Documents\glm_vectorizing
reading metrics : data/metrics
writing results : results


## Stage 1 - Extract the per-layer numbers

### Layer-wise reference metrics (ARI, P@5, silhouette) for every model

In [2]:
rows = []
for metric in ["ARI", "P5", "silh"]:
    for model in MODELS:
        mat = pd.read_excel(fio.DATA_METRICS / model / f"{metric}.xlsx").to_numpy(dtype=float)
        mean, std = np.nanmean(mat, axis=1), np.nanstd(mat, axis=1)
        for layer in range(len(mean)):
            rows.append({"metric": metric, "model": model, "layer": layer,
                         "mean": float(mean[layer]), "std": float(std[layer])})
save_table(pd.DataFrame(rows), "supplementary/layerwise_reference")

  numbers  ->  results/figure_data/supplementary/layerwise_reference.csv


WindowsPath('C:/Users/aleks/Documents/glm_vectorizing/results/figure_data/supplementary/layerwise_reference.csv')

### Layer-wise Precision@5 for the unseen viruses (forward + reverse complement)

The 4-mer baseline is layer-less and is omitted from this per-layer view.

In [3]:
viruses = ["Mriyavirales_preproc", "Egovirales_preproc",
           "REVERSECOMPLEMENTMriyavirales_preproc", "REVERSECOMPLEMENTEgovirales_preproc"]
LAYER_MODELS = [m for m in MODELS if m != "kmer"]        # 4-mer has no layers

def virus_label(v):
    if v.startswith("REVERSECOMPLEMENT"):
        return v.split("REVERSECOMPLEMENT")[-1].split("_")[0] + " (RC)"
    return v.split("_")[0]

rows = []
for virus in viruses:
    for model in LAYER_MODELS:
        path = fio.DATA_METRICS / model / f"P@5_matrix_{virus}.xlsx"
        if not path.exists():
            continue
        mat = pd.read_excel(path).set_index("layer")
        mean, std = mat.mean(axis=1), mat.std(axis=1)
        for layer in mat.index:
            rows.append({"virus": virus_label(virus), "model": model, "layer": int(layer),
                         "mean": float(mean[layer]), "std": float(std[layer])})
save_table(pd.DataFrame(rows), "supplementary/layerwise_newvirus_p5")

  numbers  ->  results/figure_data/supplementary/layerwise_newvirus_p5.csv


WindowsPath('C:/Users/aleks/Documents/glm_vectorizing/results/figure_data/supplementary/layerwise_newvirus_p5.csv')

## Stage 2 - Render one figure per model

### Reference metrics per layer

In [4]:
df = load_table("supplementary/layerwise_reference")
titles = {"ARI": "Adjusted Rand Index", "P5": "Precision@5", "silh": "Silhouette"}
for model in MODELS:
    for metric in ["ARI", "P5", "silh"]:
        d = df[(df["model"] == model) & (df["metric"] == metric)].sort_values("layer")
        fig, ax = plt.subplots()
        ax.bar(d["layer"], d["mean"], yerr=d["std"], capsize=4, color=MODEL_COLOR[model],
               error_kw={"elinewidth": 0.6, "alpha": 0.6})
        ax.set_title(f"{titles[metric]} across layers - {DISPLAY[model]}")
        ax.set_xlabel("Layer"); ax.set_ylabel("Score")
        fig.tight_layout()
        save_figure(fig, f"layerwise_{model}_{metric}", kind="supplementary")
        plt.close(fig)
print("done: reference per-layer figures")

  figure   ->  results/figures/supplementary/layerwise_kmer_ARI.{png,pdf}


  figure   ->  results/figures/supplementary/layerwise_kmer_P5.{png,pdf}


  figure   ->  results/figures/supplementary/layerwise_kmer_silh.{png,pdf}


  figure   ->  results/figures/supplementary/layerwise_dnabert2_ARI.{png,pdf}


  figure   ->  results/figures/supplementary/layerwise_dnabert2_P5.{png,pdf}


  figure   ->  results/figures/supplementary/layerwise_dnabert2_silh.{png,pdf}


... (5 outputs left out)

  figure   ->  results/figures/supplementary/layerwise_viralm_silh.{png,pdf}


  figure   ->  results/figures/supplementary/layerwise_evo2_ARI.{png,pdf}


  figure   ->  results/figures/supplementary/layerwise_evo2_P5.{png,pdf}


  figure   ->  results/figures/supplementary/layerwise_evo2_silh.{png,pdf}
done: reference per-layer figures


### Unseen-virus Precision@5 per layer

In [5]:
df = load_table("supplementary/layerwise_newvirus_p5")
for virus in df["virus"].unique():
    for model in MODELS:
        d = df[(df["virus"] == virus) & (df["model"] == model)].sort_values("layer")
        if d.empty:
            continue
        fig, ax = plt.subplots()
        ax.bar(d["layer"], d["mean"], yerr=d["std"], capsize=4, color=MODEL_COLOR[model],
               error_kw={"elinewidth": 0.6, "alpha": 0.6})
        ax.set_title(f"Precision@5 across layers - {DISPLAY[model]} ({virus})")
        ax.set_xlabel("Layer"); ax.set_ylabel("Precision@5")
        fig.tight_layout()
        slug = virus.replace(" (RC)", "_rc").replace(" ", "_")
        save_figure(fig, f"newvirus_p5_{model}_{slug}", kind="supplementary")
        plt.close(fig)
print("done: unseen-virus per-layer figures")

  figure   ->  results/figures/supplementary/newvirus_p5_dnabert2_Mriyavirales.{png,pdf}


  figure   ->  results/figures/supplementary/newvirus_p5_dnabertS_Mriyavirales.{png,pdf}


  figure   ->  results/figures/supplementary/newvirus_p5_viralm_Mriyavirales.{png,pdf}


  figure   ->  results/figures/supplementary/newvirus_p5_evo2_Mriyavirales.{png,pdf}


  figure   ->  results/figures/supplementary/newvirus_p5_dnabert2_Egovirales.{png,pdf}


  figure   ->  results/figures/supplementary/newvirus_p5_dnabertS_Egovirales.{png,pdf}


... (6 outputs left out)

  figure   ->  results/figures/supplementary/newvirus_p5_dnabert2_Egovirales_rc.{png,pdf}


  figure   ->  results/figures/supplementary/newvirus_p5_dnabertS_Egovirales_rc.{png,pdf}


  figure   ->  results/figures/supplementary/newvirus_p5_viralm_Egovirales_rc.{png,pdf}


  figure   ->  results/figures/supplementary/newvirus_p5_evo2_Egovirales_rc.{png,pdf}
done: unseen-virus per-layer figures


---
All supplementary figures written to `results/figures/supplementary/` (PNG + PDF);
their numbers live in `results/figure_data/supplementary/`.